# SolarSDE Notebook 3 — Baselines

**Runtime:** ~2-4 hours on Colab T4 / Kaggle P100

**Prerequisite:** Notebooks 1 and 2 must have run (needs latents + SolarSDE main results).

**This notebook trains 5 probabilistic baselines:**
1. Persistence (with Gaussian noise calibrated from training residuals)
2. Smart Persistence (persist the clear-sky index)
3. LSTM deterministic + calibrated Gaussian noise
4. MC-Dropout LSTM (100 stochastic forward passes at inference)
5. CSDI (conditional score-based diffusion, non-autoregressive transformer)

Each baseline is evaluated at the same 5 horizons (1, 5, 10, 20, 30 min) and combined with SolarSDE results into the main comparison table.


In [ ]:
# ==== Install dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("pvlib", "properscoring", "pyarrow", "tqdm")
print("Dependencies installed.")


In [ ]:
# ==== Environment Detection & Setup ====
import os, sys, time, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

PERSIST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_DIR = PERSIST_DIR / "latents"
LATENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")
print(f"Working directory:  {WORK_DIR}")


In [ ]:
# ==== GPU Check ====
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: Running on CPU — will be slow. Change Runtime > Change runtime type > GPU")
print(f"Using device: {DEVICE}")


## Fast-start — pull Notebook 1 outputs from GitHub (skips VAE retraining)

In [ ]:
# ==== Fast-start: fetch Notebook 1 outputs from GitHub if persistent storage is empty ====
# This lets Notebooks 2-5 run standalone without having re-executed Notebook 1.
GITHUB_REPO = "https://github.com/keshavkrishnan08/SDE"
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

need_nb1_outputs = not (CHECKPOINT_DIR / "vae_best.pt").exists() or not any(LATENT_DIR.glob("train_*.npy"))
if need_nb1_outputs:
    print("Notebook 1 outputs not found in persistent storage — pulling from GitHub ...")
    import requests
    files_to_pull = [
        ("checkpoints/vae_best.pt",          CHECKPOINT_DIR / "vae_best.pt"),
        ("checkpoints/vae_final.pt",         CHECKPOINT_DIR / "vae_final.pt"),
        ("results/vae_training_history.csv", RESULTS_DIR / "vae_training_history.csv"),
        ("splits/train.parquet",             PERSIST_DIR / "splits" / "train.parquet"),
        ("splits/val.parquet",               PERSIST_DIR / "splits" / "val.parquet"),
        ("splits/test.parquet",              PERSIST_DIR / "splits" / "test.parquet"),
    ]
    for split in ["train", "val", "test"]:
        for k in ["latents", "cti", "ghi", "covariates", "is_ramp"]:
            files_to_pull.append((f"latents/{split}_{k}.npy", LATENT_DIR / f"{split}_{k}.npy"))

    for rel, dest in files_to_pull:
        url = f"{GITHUB_RAW}/colab_outputs/{rel}"
        if dest.exists() and dest.stat().st_size > 0:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 100:
                dest.write_bytes(r.content)
                print(f"  OK  {rel}  ({len(r.content)/1e6:.2f} MB)")
            else:
                print(f"  SKIP {rel}  (status {r.status_code})")
        except Exception as e:
            print(f"  FAIL {rel}: {e}")
    print("Fast-start pull complete.")
else:
    print("Notebook 1 outputs already present in persistent storage.")


## 1. Load data + config

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm import tqdm
import time, math

def load_split(split):
    return {
        "Z":   np.load(LATENT_DIR / f"{split}_latents.npy"),
        "cti": np.load(LATENT_DIR / f"{split}_cti.npy"),
        "ghi": np.load(LATENT_DIR / f"{split}_ghi.npy"),
        "cov": np.load(LATENT_DIR / f"{split}_covariates.npy"),
        "ramp": np.load(LATENT_DIR / f"{split}_is_ramp.npy"),
    }

data = {s: load_split(s) for s in ["train", "val", "test"]}

# Load splits parquet to get clear_sky_index, ghi_clearsky for Smart Persistence
train_df = pd.read_parquet(SPLITS_DIR / "train.parquet")
val_df   = pd.read_parquet(SPLITS_DIR / "val.parquet")
test_df  = pd.read_parquet(SPLITS_DIR / "test.parquet")

HORIZONS = [6, 30, 60, 120, 180]
HORIZON_MIN = {6: 1, 30: 5, 60: 10, 120: 20, 180: 30}
N_SAMPLES = 50
SEQ_LEN = 30
N_EVAL = min(1000, len(data["test"]["ghi"]) - max(HORIZONS) - 1)
print(f"Horizons: {list(HORIZON_MIN.values())} min, MC samples: {N_SAMPLES}, Eval points: {N_EVAL}")


## 2. Shared metrics + output buffer

In [ ]:
# ==== Probabilistic forecasting metrics ====
def crps_empirical(y_true, y_samples):
    """CRPS from empirical samples. y_true: (N,), y_samples: (N, M)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    y_med = np.median(y_samples, axis=1)
    y_range = y_true.max() - y_true.min()
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps": float(crps.mean()),
        "picp": picp(y_true, y_samples, alpha),
        "pinaw": pinaw(y_samples, y_range, alpha),
        "rmse": rmse(y_true, y_med),
        "mae":  mae(y_true, y_med),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    return out


In [ ]:
all_baseline_results = {}

def save_baseline(name, results_by_h):
    df = pd.DataFrame.from_dict(results_by_h, orient="index").sort_values("horizon_min")
    df["model"] = name
    df.to_csv(RESULTS_DIR / f"baseline_{name}_results.csv", index=False)
    all_baseline_results[name] = df
    print(f"\n{name} results:")
    print(df[["horizon_min", "crps", "rmse", "mae", "picp", "pinaw"]].to_string())


## 3. Persistence baseline

In [ ]:
# Calibrate per-horizon Gaussian noise std on TRAIN residuals
print("Calibrating Persistence noise ...")
tr_ghi = data["train"]["ghi"]
noise_std = {}
for h in HORIZONS:
    r = tr_ghi[h:] - tr_ghi[:-h]
    noise_std[h] = float(np.std(r))
    print(f"  horizon {HORIZON_MIN[h]} min: residual std = {noise_std[h]:.2f} W/m²")

te_ghi = data["test"]["ghi"]; te_ramp = data["test"]["ramp"]
rng = np.random.default_rng(42)
res_pers = {}
for h in HORIZONS:
    yt, ys, rm = [], [], []
    for i in range(N_EVAL):
        if i + h < len(te_ghi):
            y_pred = te_ghi[i]
            samples = np.clip(y_pred + rng.normal(0, noise_std[h], size=N_SAMPLES), 0, None)
            yt.append(te_ghi[i + h]); ys.append(samples); rm.append(te_ramp[i + h])
    yt = np.array(yt); ys = np.array(ys); rm = np.array(rm)
    m = all_metrics(yt, ys, is_ramp=rm); m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
    res_pers[h] = m
save_baseline("persistence", res_pers)


## 4. Smart Persistence (persist clear-sky index)

In [ ]:
# Need clear_sky_index and ghi_clearsky aligned to latent/test arrays
te_kt = test_df["clear_sky_index"].values.astype(np.float32)
te_gcs = test_df["ghi_clearsky"].values.astype(np.float32)
tr_kt = train_df["clear_sky_index"].values.astype(np.float32)
tr_gcs = train_df["ghi_clearsky"].values.astype(np.float32)
tr_ghi_df = train_df["ghi"].values.astype(np.float32)

print("Calibrating Smart Persistence noise ...")
sp_std = {}
for h in HORIZONS:
    pred_tr = tr_kt[:-h] * tr_gcs[h:]
    act_tr = tr_ghi_df[h:]
    sp_std[h] = float(np.std(act_tr - pred_tr))
    print(f"  horizon {HORIZON_MIN[h]} min: SP residual std = {sp_std[h]:.2f} W/m²")

rng = np.random.default_rng(42)
res_sp = {}
for h in HORIZONS:
    yt, ys, rm = [], [], []
    for i in range(N_EVAL):
        j = i + h
        if j < len(te_ghi) and j < len(te_gcs):
            pt = te_kt[i] * te_gcs[j]
            samples = np.clip(pt + rng.normal(0, sp_std[h], size=N_SAMPLES), 0, None)
            yt.append(te_ghi[j]); ys.append(samples); rm.append(te_ramp[j])
    yt = np.array(yt); ys = np.array(ys); rm = np.array(rm)
    m = all_metrics(yt, ys, is_ramp=rm); m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
    res_sp[h] = m
save_baseline("smart_persistence", res_sp)


## 5. Deterministic LSTM + calibrated Gaussian noise

In [ ]:
# Build sequence dataset from (GHI, kt, zenith, covariates)
def build_seq_tensors(df, data_split, seq_len, horizons):
    # features: ghi, clear_sky_index, solar_zenith, temperature, humidity, wind_speed
    f_cols = ["ghi", "clear_sky_index", "solar_zenith"]
    for c in ["temperature", "humidity", "wind_speed"]:
        if c in df.columns: f_cols.append(c)
    X_arr = df[f_cols].fillna(0).values.astype(np.float32)
    ghi = df["ghi"].values.astype(np.float32)
    mx = max(horizons)
    Xs, Ys = [], []
    for i in range(seq_len, len(X_arr) - mx):
        Xs.append(X_arr[i - seq_len:i])
        Ys.append(np.array([ghi[i + h] for h in horizons], dtype=np.float32))
    return torch.tensor(np.stack(Xs)), torch.tensor(np.stack(Ys))

Xtr, Ytr = build_seq_tensors(train_df, data["train"], SEQ_LEN, HORIZONS)
Xva, Yva = build_seq_tensors(val_df,   data["val"],   SEQ_LEN, HORIZONS)
Xte, Yte = build_seq_tensors(test_df,  data["test"],  SEQ_LEN, HORIZONS)
print(f"Seq shapes: train={Xtr.shape}/{Ytr.shape}, val={Xva.shape}, test={Xte.shape}")

# Normalize features based on train stats
mu_f = Xtr.mean(dim=(0,1), keepdim=True); sd_f = Xtr.std(dim=(0,1), keepdim=True) + 1e-6
Xtr_n = (Xtr - mu_f) / sd_f; Xva_n = (Xva - mu_f) / sd_f; Xte_n = (Xte - mu_f) / sd_f
INPUT_DIM = Xtr_n.shape[-1]; N_H = len(HORIZONS)

class LSTMF(nn.Module):
    def __init__(self, d_in, h=128, nl=2, n_out=5, drop=0.0):
        super().__init__()
        self.lstm = nn.LSTM(d_in, h, nl, batch_first=True, dropout=drop if nl > 1 else 0.0)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(h, n_out)
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(self.drop(hn[-1]))

def train_lstm(model, X, Y, Xv, Yv, epochs=40, bs=128, lr=1e-3, tag=""):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); crit = nn.MSELoss()
    ds = TensorDataset(X, Y); dl = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=True)
    dv = DataLoader(TensorDataset(Xv, Yv), batch_size=bs)
    best = float("inf")
    for ep in range(1, epochs + 1):
        model.train(); tl = 0; n = 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item(); n += 1
        tl /= n
        model.eval(); vl = vn = 0
        with torch.no_grad():
            for xb, yb in dv:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl += crit(model(xb), yb).item(); vn += 1
        vl /= max(vn, 1)
        if vl < best: best = vl; torch.save(model.state_dict(), CHECKPOINT_DIR / f"{tag}_best.pt")
        if ep % 10 == 0 or ep == 1:
            print(f"  {tag} epoch {ep}/{epochs}: train={tl:.4f} val={vl:.4f}")
    model.load_state_dict(torch.load(CHECKPOINT_DIR / f"{tag}_best.pt", map_location=DEVICE))
    return model

print("Training deterministic LSTM ...")
torch.manual_seed(42)
lstm = train_lstm(LSTMF(INPUT_DIM, 128, 2, N_H, drop=0.0), Xtr_n, Ytr, Xva_n, Yva, epochs=30, tag="lstm_det")

# Calibrate noise from train residuals
lstm.eval()
with torch.no_grad():
    pred_tr = lstm(Xtr_n.to(DEVICE)).cpu().numpy()
res_tr = Ytr.numpy() - pred_tr
lstm_std = {HORIZONS[i]: float(res_tr[:, i].std()) for i in range(N_H)}
for h, s in lstm_std.items():
    print(f"  horizon {HORIZON_MIN[h]} min: LSTM residual std = {s:.2f}")

# Evaluate with Gaussian noise
rng = np.random.default_rng(42)
with torch.no_grad():
    pred_te = lstm(Xte_n.to(DEVICE)).cpu().numpy()  # (N, 5)
# Build target arrays aligned with sequence dataset
te_ghi_seq = test_df["ghi"].values.astype(np.float32)
te_ramp_seq = test_df["is_ramp"].values.astype(bool)

res_lstm = {}
for hi, h in enumerate(HORIZONS):
    yt, ys, rm = [], [], []
    for i in range(min(N_EVAL, len(pred_te))):
        target_idx = SEQ_LEN + i + h
        if target_idx < len(te_ghi_seq):
            pt = pred_te[i, hi]
            samples = np.clip(pt + rng.normal(0, lstm_std[h], size=N_SAMPLES), 0, None)
            yt.append(te_ghi_seq[target_idx]); ys.append(samples); rm.append(te_ramp_seq[target_idx])
    yt = np.array(yt); ys = np.array(ys); rm = np.array(rm)
    m = all_metrics(yt, ys, is_ramp=rm); m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
    res_lstm[h] = m
save_baseline("lstm", res_lstm)


## 6. MC-Dropout LSTM

In [ ]:
print("Training MC-Dropout LSTM ...")
torch.manual_seed(42)
mcd = train_lstm(LSTMF(INPUT_DIM, 128, 2, N_H, drop=0.1), Xtr_n, Ytr, Xva_n, Yva, epochs=30, tag="lstm_mcd")

# MC inference: keep dropout active, run 50 forward passes
def mc_predict(model, X, n_passes=50, bs=256):
    model.train()  # keep dropout on
    out = []
    for _ in range(n_passes):
        preds = []
        with torch.no_grad():
            for i in range(0, len(X), bs):
                preds.append(model(X[i:i+bs].to(DEVICE)).cpu())
        out.append(torch.cat(preds, dim=0).numpy())
    model.eval()
    return np.stack(out, axis=0)  # (passes, N, H)

print("MC sampling on test set ...")
mc_pred = mc_predict(mcd, Xte_n, n_passes=N_SAMPLES)  # (N_SAMPLES, N, H)

res_mcd = {}
for hi, h in enumerate(HORIZONS):
    yt, ys, rm = [], [], []
    for i in range(min(N_EVAL, mc_pred.shape[1])):
        target_idx = SEQ_LEN + i + h
        if target_idx < len(te_ghi_seq):
            samples = np.clip(mc_pred[:, i, hi], 0, None)
            yt.append(te_ghi_seq[target_idx]); ys.append(samples); rm.append(te_ramp_seq[target_idx])
    yt = np.array(yt); ys = np.array(ys); rm = np.array(rm)
    m = all_metrics(yt, ys, is_ramp=rm); m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
    res_mcd[h] = m
save_baseline("mc_dropout", res_mcd)


## 7. CSDI (conditional score-based diffusion)

In [ ]:
class DiffEmb(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        half = d // 2
        emb = math.log(10000) / (half - 1)
        self.register_buffer("emb", torch.exp(torch.arange(half).float() * -emb))
    def forward(self, t):
        e = t.unsqueeze(-1).float() * self.emb.unsqueeze(0)
        return torch.cat([e.sin(), e.cos()], dim=-1)

class TxBlock(nn.Module):
    def __init__(self, d=64, nh=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, nh, batch_first=True)
        self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, d * 4), nn.GELU(), nn.Linear(d * 4, d))
    def forward(self, x):
        h = self.n1(x); h, _ = self.attn(h, h, h); x = x + h
        return x + self.ffn(self.n2(x))

class CSDIScoreNet(nn.Module):
    def __init__(self, d_in, d=64, nh=4, nl=4, steps=100):
        super().__init__()
        self.steps = steps
        self.demb = DiffEmb(d)
        self.proj = nn.Linear(d_in + 1, d)
        self.dproj = nn.Linear(d, d)
        self.blocks = nn.ModuleList([TxBlock(d, nh) for _ in range(nl)])
        self.out = nn.Linear(d, 1)
        betas = torch.linspace(1e-4, 0.02, steps); alphas = 1 - betas; ac = torch.cumprod(alphas, 0)
        self.register_buffer("betas", betas); self.register_buffer("alphas", alphas); self.register_buffer("ac", ac)
        self.register_buffer("sac", torch.sqrt(ac)); self.register_buffer("s1mac", torch.sqrt(1 - ac))

    def _forward(self, x_cond, y_noisy, t_idx):
        B, S, D = x_cond.shape
        # Append a "prediction slot" token with y_noisy as its first feature
        extra = torch.zeros(B, 1, D, device=x_cond.device)
        extra[:, 0, 0] = y_noisy.squeeze(-1)
        seq = torch.cat([x_cond, extra], dim=1)
        tgt_chan = torch.zeros(B, S + 1, 1, device=x_cond.device)
        tgt_chan[:, -1, 0] = y_noisy.squeeze(-1)
        h = self.proj(torch.cat([seq, tgt_chan], dim=-1))
        te = self.demb(t_idx.float()); h = h + self.dproj(te).unsqueeze(1)
        for blk in self.blocks: h = blk(h)
        return self.out(h[:, -1, :])

    def training_loss(self, x_cond, y):
        B = y.shape[0]; dev = y.device
        t_idx = torch.randint(0, self.steps, (B,), device=dev)
        eps = torch.randn_like(y.unsqueeze(-1))
        y_noisy = self.sac[t_idx].unsqueeze(-1) * y.unsqueeze(-1) + self.s1mac[t_idx].unsqueeze(-1) * eps
        pred = self._forward(x_cond, y_noisy, t_idx)
        return F.mse_loss(pred, eps)

    @torch.no_grad()
    def sample(self, x_cond, n=50):
        B = x_cond.shape[0]; dev = x_cond.device
        xc = x_cond.unsqueeze(1).expand(B, n, -1, -1).reshape(B * n, *x_cond.shape[1:])
        x = torch.randn(B * n, 1, device=dev)
        for i in reversed(range(self.steps)):
            ti = torch.full((B * n,), i, device=dev, dtype=torch.long)
            eps_p = self._forward(xc, x, ti)
            b, a, ab = self.betas[i], self.alphas[i], self.ac[i]
            x = (1 / a.sqrt()) * (x - b / (1 - ab).sqrt() * eps_p)
            if i > 0: x = x + b.sqrt() * torch.randn_like(x)
        return x.squeeze(-1).view(B, n)

# Train CSDI on horizon-specific targets (we'll train one model and use horizon-index conditioning
# For simplicity, use the first horizon (6 steps = 1 min) target as training target.
# For multi-horizon, we'd need either separate models or horizon-conditioning.
# To keep compute manageable, we train one CSDI per group of horizons.
print("Training CSDI (using first horizon target for joint training) ...")
torch.manual_seed(42)
csdi = CSDIScoreNet(d_in=INPUT_DIM, d=64, nh=4, nl=4, steps=50).to(DEVICE)
opt = torch.optim.Adam(csdi.parameters(), lr=1e-3)
ds = TensorDataset(Xtr_n, Ytr[:, 0])
dl = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)

EPOCHS_CSDI = 30
t0 = time.time()
for ep in range(1, EPOCHS_CSDI + 1):
    csdi.train(); tl = 0; n = 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        l = csdi.training_loss(xb, yb)
        opt.zero_grad(); l.backward(); opt.step()
        tl += l.item(); n += 1
    if ep % 5 == 0 or ep == 1:
        print(f"  CSDI epoch {ep}/{EPOCHS_CSDI}: train={tl/n:.4f}  time={(time.time()-t0)/60:.1f}min")
torch.save(csdi.state_dict(), CHECKPOINT_DIR / "csdi_best.pt")

# Evaluate CSDI on all horizons (note: we trained on horizon 6; for other horizons we retrain briefly)
res_csdi = {}

# Quick multi-horizon: re-finetune briefly per horizon
for hi, h in enumerate(HORIZONS):
    # Create targeted dataset
    ds_h = TensorDataset(Xtr_n, Ytr[:, hi])
    dl_h = DataLoader(ds_h, batch_size=128, shuffle=True, drop_last=True)
    # Warm start from base
    csdi_h = CSDIScoreNet(d_in=INPUT_DIM, d=64, nh=4, nl=4, steps=50).to(DEVICE)
    csdi_h.load_state_dict(csdi.state_dict())
    opt_h = torch.optim.Adam(csdi_h.parameters(), lr=5e-4)
    for ep in range(10):  # short fine-tune
        csdi_h.train()
        for xb, yb in dl_h:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            l = csdi_h.training_loss(xb, yb)
            opt_h.zero_grad(); l.backward(); opt_h.step()
    csdi_h.eval()
    print(f"  CSDI h={HORIZON_MIN[h]}min: generating samples ...")

    yt, ys, rm = [], [], []
    bs = 8
    for i in range(0, min(N_EVAL, len(Xte_n)), bs):
        xb = Xte_n[i:i+bs].to(DEVICE)
        with torch.no_grad():
            samp = csdi_h.sample(xb, n=N_SAMPLES).cpu().numpy()  # (B, N)
        for k in range(samp.shape[0]):
            target_idx = SEQ_LEN + i + k + h
            if target_idx < len(te_ghi_seq):
                yt.append(te_ghi_seq[target_idx])
                ys.append(np.clip(samp[k], 0, None))
                rm.append(te_ramp_seq[target_idx])
    yt = np.array(yt); ys = np.array(ys); rm = np.array(rm)
    m = all_metrics(yt, ys, is_ramp=rm); m["horizon_min"] = HORIZON_MIN[h]; m["horizon_steps"] = h; m["n_eval"] = len(yt)
    res_csdi[h] = m

save_baseline("csdi", res_csdi)


## 8. Combined main results table

In [ ]:
# Load SolarSDE results from Notebook 2
solar = pd.read_csv(RESULTS_DIR / "solar_sde_main_results.csv")
solar["model"] = "SolarSDE"

# Stack all
parts = [solar]
for name in ["persistence", "smart_persistence", "lstm", "mc_dropout", "csdi"]:
    parts.append(all_baseline_results[name])

combined = pd.concat(parts, ignore_index=True)
combined = combined[["model", "horizon_min", "crps", "rmse", "mae", "picp", "pinaw", "ramp_crps"]]
combined = combined.sort_values(["model", "horizon_min"]).reset_index(drop=True)

combined.to_csv(RESULTS_DIR / "main_results_combined.csv", index=False)

print("=" * 80)
print("MAIN RESULTS (all models, all horizons)")
print("=" * 80)
print(combined.to_string(index=False))

# Pivot for paper-style table at 10-min horizon
h_focus = 10
pivot = combined[combined["horizon_min"] == h_focus].set_index("model")
pivot = pivot[["crps", "rmse", "mae", "picp", "pinaw", "ramp_crps"]]
pivot.to_csv(RESULTS_DIR / f"main_table_h{h_focus}min.csv")

print(f"\n{'=' * 80}")
print(f"Main Table (h = {h_focus} minutes)")
print("=" * 80)
print(pivot.to_string())

# Compute skill scores vs persistence
pers_crps = {r["horizon_min"]: r["crps"] for _, r in
             combined[combined["model"] == "persistence"].iterrows()}
combined["skill_vs_persistence"] = combined.apply(
    lambda r: 1 - r["crps"] / pers_crps[r["horizon_min"]], axis=1)
combined.to_csv(RESULTS_DIR / "main_results_combined.csv", index=False)

print(f"\n{'=' * 80}")
print("Skill Scores vs Persistence (higher = better)")
print("=" * 80)
skill_pivot = combined.pivot(index="model", columns="horizon_min", values="skill_vs_persistence")
print(skill_pivot.to_string())


In [ ]:
# ==== Zip outputs and prepare download ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path}")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive size: {size_mb:.1f} MB")

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(zip_path))
        print("Download triggered (check browser).")
    except Exception as e:
        print(f"Auto-download unavailable: {e}. Download manually from {zip_path}")
else:
    print(f"Archive at: {zip_path}  — download via Kaggle output tab or file browser.")


In [ ]:
print("=" * 70)
print("NOTEBOOK 3 COMPLETE")
print("=" * 70)
print("Baselines trained: Persistence, Smart Persistence, LSTM, MC-Dropout, CSDI")
print(f"Combined main table: {RESULTS_DIR / 'main_results_combined.csv'}")
print("Next: 04_ablations.ipynb")
